# 01 · Train the GRU autoencoder

Trains the Bidirectional GRU autoencoder on clean 10-step windows of `[latitude, longitude, speed, course, heading]` from the training vessels, validated on a separate set of vessels.

**Input:** the Baltic AIS CSV at `MARITIME_DATA_CSV` (default `data/training_set.csv`).  
**Output** (in `MARITIME_MODEL_DIR`, default `models/latest-model/`):
- `final_data_scaler.pkl`: the StandardScaler the encoder was trained with. Notebook 02 and `app/api.py` load it.
- `final_autoencoder_model.keras`: the full autoencoder.
- `gru_encoder.keras`: the encoder half (input → 128-d latent vector).

Run this before `02_train_detector.ipynb`.

In [ ]:
import sys
from pathlib import Path

# Make the shared maritime_ad package importable when running from notebooks/
ROOT = Path.cwd().resolve()
if not (ROOT / "maritime_ad").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import joblib
import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.layers import GRU, Bidirectional, Dense, Input, RepeatVector, TimeDistributed
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam

from maritime_ad import config
from maritime_ad.data import load_ais, make_windows, scale_windows, split_by_vessel

np.random.seed(config.SEED)
tf.random.set_seed(config.SEED)
config.MODEL_DIR.mkdir(parents=True, exist_ok=True)
config.RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# Training hyperparameters
EPOCHS = 150
BATCH_SIZE = 256
LEARNING_RATE = 1e-3
GRU_UNITS = 64          # per direction, so the latent vector is 128-d
NOISE_FACTOR = 0.05     # denoising: train on noisy input, reconstruct the clean window

print("Data:  ", config.DATA_CSV)
print("Models:", config.MODEL_DIR)

In [ ]:
# Load, then split by vessel so no vessel's windows appear in more than one split
df = load_ais(config.DATA_CSV)
df_train, df_val, _ = split_by_vessel(df)  # the test vessels are only used in notebook 02
del df

X_train, _ = make_windows(df_train)
X_val, _ = make_windows(df_val)
print(f"Vessels: train {df_train['mmsi'].nunique()}, val {df_val['mmsi'].nunique()}")
print(f"Windows: train {X_train.shape}, val {X_val.shape}")

In [ ]:
# Fit the scaler on training windows only, and save it for notebook 02 and the API
scaler = StandardScaler().fit(X_train.reshape(-1, X_train.shape[-1]))
joblib.dump(scaler, config.MODEL_DIR / config.SCALER_FILE)

X_train_scaled = scale_windows(X_train, scaler)
X_val_scaled = scale_windows(X_val, scaler)
print("Scaler saved to", config.MODEL_DIR / config.SCALER_FILE)

In [ ]:
inputs = Input(shape=(config.SEQ_LEN, len(config.FEATURES)), name="window")
latent = Bidirectional(GRU(GRU_UNITS, activation="relu"), name="encoder")(inputs)
repeated = RepeatVector(config.SEQ_LEN)(latent)
decoded = Bidirectional(GRU(GRU_UNITS, activation="relu", return_sequences=True), name="decoder")(repeated)
outputs = TimeDistributed(Dense(len(config.FEATURES), activation="linear"))(decoded)

autoencoder = Model(inputs, outputs, name="gru_autoencoder")
autoencoder.compile(optimizer=Adam(learning_rate=LEARNING_RATE), loss="mean_squared_error")
autoencoder.summary()

rng = np.random.default_rng(config.SEED)
X_train_noisy = X_train_scaled + NOISE_FACTOR * rng.standard_normal(X_train_scaled.shape, dtype=np.float32)

history = autoencoder.fit(
    X_train_noisy,
    X_train_scaled,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    validation_data=(X_val_scaled, X_val_scaled),
    callbacks=[
        ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=5, verbose=1),
        EarlyStopping(monitor="val_loss", patience=15, restore_best_weights=True, verbose=1),
    ],
    verbose=1,
)

In [ ]:
# EarlyStopping restored the best weights; save the autoencoder and its encoder half
encoder = Model(inputs, latent, name="gru_encoder")
autoencoder.save(config.MODEL_DIR / config.AUTOENCODER_FILE)
encoder.save(config.MODEL_DIR / config.ENCODER_FILE)

val_loss = history.history["val_loss"]
best_epoch = int(np.argmin(val_loss)) + 1
print(f"Best epoch {best_epoch} of {len(val_loss)}, val_loss {min(val_loss):.5f}")

plt.figure(figsize=(8, 5))
plt.plot(history.history["loss"], label="train")
plt.plot(val_loss, label="validation")
plt.axvline(best_epoch - 1, color="grey", linestyle="--", label=f"best (epoch {best_epoch})")
plt.xlabel("Epoch")
plt.ylabel("MSE")
plt.title("Autoencoder training loss")
plt.legend()
plt.savefig(config.RESULTS_DIR / "autoencoder_loss.png", bbox_inches="tight")
plt.show()